# 1. Generate diffraction simulations (with per-frame thickness)

This notebook keeps the existing simulation pipeline and adds per-frame crystal thickness for abTEM multislice.
Thickness can come from an external table or be randomly generated from mean/spread.

In [ ]:
# %%
from pathlib import Path
import csv
import logging

import numpy as np
from diffpy.structure import Lattice, Structure
from diffpy.structure.spacegroups import GetSpaceGroup
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.crystal_map import Phase
from orix.sampling import get_sample_reduced_fundamental

from parse_pdb_with_scale_remove_h import parse_pdb_with_scale

# ── user-tunable parameters ────────────────────────────────────────────────────
# input = "/home/bubl3932/files/simulations/aP_lyso/lyso.pdb"
# input = "/home/bubl3932/files/simulations/cP_LTA/7108314.cif"
# input = "/home/bubl3932/files/simulations/MFM300-VIII_tI/4135627.cif"
# input = "/home/bubl3932/files/simulations/MFM300-VIII_tI/4135627.cif"
input = "/Users/xiaodong/Desktop/simulations/MFM300-VIII_tI/MFM300-VIII.cif"

REF_PATH = Path(input).expanduser()
RESOLUTION_DEG = 10
MAX_RESOLUTION = 0.5
DIRECT_BEAM = False
MAX_EXC_ERR_AINV = 0.02
SHAPE_FACTOR_W_AINV = 0.001
VOLTAGE_KV = 300

# ── dynamical intensities via abTEM multislice ─────────────────────────────────
USE_ABTEM_MULTISLICE = True
ABTEM_SAMPLING_A = 0.10
ABTEM_SLICE_THICKNESS_A = 1.0
ABTEM_PROJECTION = "infinite"
ABTEM_PARAMETRIZATION = "lobato"
ABTEM_REMOVE_LOW_INTENSITY = 1e-6

# ── per-frame thickness configuration (nm) ─────────────────────────────────────
# Choose one: "table" or "random"
THICKNESS_SOURCE = "random"

# table mode
THICKNESS_TABLE_PATH = REF_PATH.parent / "thickness_nm.csv"
THICKNESS_TABLE_FRAME_COL = "frame"
THICKNESS_TABLE_VALUE_COL = "thickness_nm"

# random mode (normal distribution, clipped)
RANDOM_THICKNESS_MEAN_NM = 210.0
RANDOM_THICKNESS_STD_NM = 80.0
RANDOM_THICKNESS_MIN_NM = 40.0
RANDOM_THICKNESS_MAX_NM = 650.0
RANDOM_THICKNESS_SEED = 42
WRITE_RANDOM_TABLE = True
RANDOM_TABLE_OUT_PATH = REF_PATH.parent / "thickness_nm_random.csv"
# ───────────────────────────────────────────────────────────────────────────────

def _write_thickness_table_nm(path: Path, thickness_nm: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["frame", "thickness_nm"])
        for i, t in enumerate(thickness_nm):
            writer.writerow([int(i), float(t)])


def _load_thickness_table_nm(
    path: Path,
    n_frames: int,
    frame_col: str = "frame",
    value_col: str = "thickness_nm",
) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Thickness table not found: {path}")

    with open(path, "r", newline="") as fh:
        first_line = fh.readline()

    has_header = any(ch.isalpha() for ch in first_line)

    if has_header:
        with open(path, "r", newline="") as fh:
            reader = csv.DictReader(fh)
            fieldnames = set(reader.fieldnames or [])
            rows = list(reader)

        if value_col not in fieldnames:
            raise ValueError(
                f"Header mode requires column '{value_col}'. Columns: {sorted(fieldnames)}"
            )

        if frame_col in fieldnames:
            out = np.full(n_frames, np.nan, dtype=float)
            for row in rows:
                idx = int(row[frame_col])
                if idx < 0 or idx >= n_frames:
                    continue
                out[idx] = float(row[value_col])
            if np.isnan(out).any():
                missing = int(np.isnan(out).sum())
                raise ValueError(
                    f"Thickness table missing {missing} frame(s). Add rows for all frame indices 0..{n_frames - 1}."
                )
            thickness_nm = out
        else:
            thickness_nm = np.array([float(row[value_col]) for row in rows], dtype=float)
            if thickness_nm.size != n_frames:
                raise ValueError(
                    f"Thickness table has {thickness_nm.size} values, expected {n_frames}."
                )
    else:
        raw = np.loadtxt(path, delimiter=",")
        if raw.ndim == 1:
            thickness_nm = raw.astype(float)
            if thickness_nm.size != n_frames:
                raise ValueError(
                    f"Thickness list has {thickness_nm.size} values, expected {n_frames}."
                )
        else:
            if raw.shape[1] < 2:
                raise ValueError("Numeric table must have either 1 column (values) or 2 columns (frame,value).")
            out = np.full(n_frames, np.nan, dtype=float)
            for frame_idx, value in raw[:, :2]:
                idx = int(frame_idx)
                if idx < 0 or idx >= n_frames:
                    continue
                out[idx] = float(value)
            if np.isnan(out).any():
                missing = int(np.isnan(out).sum())
                raise ValueError(
                    f"Numeric thickness table missing {missing} frame(s)."
                )
            thickness_nm = out

    if not np.isfinite(thickness_nm).all():
        raise ValueError("Thickness values contain non-finite numbers.")
    if (thickness_nm <= 0).any():
        raise ValueError("All thickness values must be > 0.")

    return thickness_nm


def _build_thickness_nm(n_frames: int) -> np.ndarray:
    source = str(THICKNESS_SOURCE).strip().lower()
    if source == "table":
        thickness_nm = _load_thickness_table_nm(
            path=Path(THICKNESS_TABLE_PATH),
            n_frames=n_frames,
            frame_col=THICKNESS_TABLE_FRAME_COL,
            value_col=THICKNESS_TABLE_VALUE_COL,
        )
    elif source == "random":
        rng = np.random.default_rng(RANDOM_THICKNESS_SEED)
        thickness_nm = rng.normal(
            loc=float(RANDOM_THICKNESS_MEAN_NM),
            scale=float(RANDOM_THICKNESS_STD_NM),
            size=n_frames,
        )
        thickness_nm = np.clip(
            thickness_nm,
            float(RANDOM_THICKNESS_MIN_NM),
            float(RANDOM_THICKNESS_MAX_NM),
        )
        if WRITE_RANDOM_TABLE:
            _write_thickness_table_nm(Path(RANDOM_TABLE_OUT_PATH), thickness_nm)
            logging.info("Wrote random thickness table: %s", RANDOM_TABLE_OUT_PATH)
    else:
        raise ValueError("THICKNESS_SOURCE must be 'table' or 'random'.")

    return thickness_nm.astype(float)


logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

if REF_PATH.suffix == ".pdb":
    cell, sg_sym, atoms = parse_pdb_with_scale(
        REF_PATH, remove_hydrogens=True, include_occupancy=False
    )
    phase = Phase(
        space_group=GetSpaceGroup(sg_sym),
        structure=Structure(atoms, Lattice(*cell)),
    )
elif REF_PATH.suffix == ".cif":
    phase = Phase.from_cif(REF_PATH)
else:
    raise ValueError("Unsupported file format. Please provide a .pdb or .cif file.")

orientations = get_sample_reduced_fundamental(
    resolution=RESOLUTION_DEG,
    point_group=phase.point_group,
)

generator = SimulationGenerator(
    accelerating_voltage=VOLTAGE_KV,
    shape_factor_model="atanc",
    approximate_precession=False,
)

sims = generator.calculate_diffraction2d(
    phase=phase,
    rotation=orientations,
    reciprocal_radius=1 / MAX_RESOLUTION,
    with_direct_beam=DIRECT_BEAM,
    max_excitation_error=MAX_EXC_ERR_AINV,
    shape_factor_width=SHAPE_FACTOR_W_AINV,
    debye_waller_factors=None,
    show_progressbar=True,
)

thickness_nm = _build_thickness_nm(sims.current_size)
logging.info(
    "Thickness summary (nm): n=%d, min=%.2f, mean=%.2f, max=%.2f",
    thickness_nm.size,
    float(thickness_nm.min()),
    float(thickness_nm.mean()),
    float(thickness_nm.max()),
)

sims.plot()
logging.info("Generated %d patterns.", sims.current_size)

MFM300-VIII: 100%|██████████| 91/91 [00:00<00:00, 140.72it/s]


FileNotFoundError: Thickness table not found: /Users/xiaodong/Desktop/simulations/MFM300-VIII_tI/thickness_nm.csv

# 2. Write simulated patterns into HDF5 and helper files

In [ ]:
# %%
import h5py
from tqdm import tqdm

from calculate_calibration import calculate_calibration
from clen_for_resolution import clen_for_dmin
from compute_B import compute_B
from create_empty_backgrounds import create_empty_backgrounds
from electron_wavelength import electron_wavelength
from generate_cell import write_cell_file
from generate_geom import write_geom_file
from helper_functions_UB import copy_h5_file, get_next_simulation_folder


def _g_per_pixel(wavelength_A: float, clen_m: float, pixels_per_m: float) -> float:
    return 1.0 / (clen_m * wavelength_A * pixels_per_m)


def _rasterize_spots_to_image(
    positions_xy_ainv: np.ndarray,
    intensities: np.ndarray,
    shape: tuple[int, int],
    beam_pos_rc: tuple[int, int],
    g_per_pixel: float,
    sigma_pix: float,
    in_plane_angle_deg: float,
) -> np.ndarray:
    img = np.zeros(shape, dtype=np.float32)
    if positions_xy_ainv.size == 0:
        return img

    ang = np.deg2rad(in_plane_angle_deg)
    c, s = np.cos(ang), np.sin(ang)
    gx = positions_xy_ainv[:, 0].astype(np.float32)
    gy = positions_xy_ainv[:, 1].astype(np.float32)
    gx_r = c * gx - s * gy
    gy_r = s * gx + c * gy

    col = beam_pos_rc[1] + gx_r / g_per_pixel
    row = beam_pos_rc[0] - gy_r / g_per_pixel

    radius = max(1, int(np.ceil(4.0 * sigma_pix)))
    rr = np.arange(-radius, radius + 1, dtype=np.float32)
    xx, yy = np.meshgrid(rr, rr, indexing="xy")
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * float(sigma_pix) ** 2)).astype(np.float32)

    for r0, c0, intensity in zip(row, col, intensities.astype(np.float32), strict=False):
        if intensity <= 0:
            continue
        r_int = int(np.round(r0))
        c_int = int(np.round(c0))

        r1 = r_int - radius
        r2 = r_int + radius + 1
        c1 = c_int - radius
        c2 = c_int + radius + 1

        kr1 = 0
        kc1 = 0
        kr2 = kernel.shape[0]
        kc2 = kernel.shape[1]

        if r1 < 0:
            kr1 = -r1
            r1 = 0
        if c1 < 0:
            kc1 = -c1
            c1 = 0
        if r2 > shape[0]:
            kr2 -= r2 - shape[0]
            r2 = shape[0]
        if c2 > shape[1]:
            kc2 -= c2 - shape[1]
            c2 = shape[1]

        if r1 >= r2 or c1 >= c2:
            continue

        img[r1:r2, c1:c2] += intensity * kernel[kr1:kr2, kc1:kc2]

    return img


def _abtem_multislice_pattern(
    ref_path: Path,
    rotation_matrix_3x3: np.ndarray,
    thickness_A: float,
    voltage_kV: float,
    max_resolution_A: float,
    direct_beam: bool,
    sampling_A: float,
    slice_thickness_A: float,
    projection: str,
    parametrization: str,
    remove_low_intensity: float,
    shape: tuple[int, int],
    beam_pos_rc: tuple[int, int],
    g_per_pixel: float,
    sigma_pix: float,
    in_plane_angle_deg: float,
) -> np.ndarray:
    import abtem
    from ase.io import read as ase_read

    unit = ase_read(str(ref_path))

    try:
        del unit[[i for i, a in enumerate(unit) if a.symbol == "H"]]
    except Exception:
        pass

    c_len = float(unit.cell.lengths()[2])
    if c_len <= 0:
        raise ValueError("ASE cell has invalid c length; cannot build thickness supercell.")

    nz = max(1, int(np.ceil(float(thickness_A) / c_len)))
    thick = unit * (1, 1, nz)

    R = np.asarray(rotation_matrix_3x3, dtype=float)
    center = (np.array([0.5, 0.5, 0.5]) @ thick.cell.array).astype(float)
    pos = thick.get_positions()
    pos = (pos - center) @ R.T + center
    thick.set_positions(pos)
    thick.set_cell(thick.cell.array @ R.T, scale_atoms=False)
    thick.wrap()

    unit_r = unit.copy()
    center_u = (np.array([0.5, 0.5, 0.5]) @ unit_r.cell.array).astype(float)
    pos_u = unit_r.get_positions()
    pos_u = (pos_u - center_u) @ R.T + center_u
    unit_r.set_positions(pos_u)
    unit_r.set_cell(unit_r.cell.array @ R.T, scale_atoms=False)
    unit_r.wrap()

    potential = abtem.Potential(
        thick,
        sampling=sampling_A,
        slice_thickness=slice_thickness_A,
        projection=projection,
        parametrization=parametrization,
    )
    wave = abtem.PlaneWave(energy=float(voltage_kV) * 1e3)
    wave.grid.match(potential)

    dp = wave.multislice(potential).diffraction_patterns().compute()
    crop_mrad = 1000.0 * float(electron_wavelength(voltage_kV)) * (1.0 / float(max_resolution_A))
    dp = dp.crop(crop_mrad)
    indexed = dp.index_diffraction_spots(cell=unit_r.cell)

    hkls = indexed.miller_indices
    pos3 = indexed.positions
    inten = np.asarray(indexed.array).squeeze()

    if not direct_beam:
        mask0 = np.all(hkls == np.array([0, 0, 0]), axis=1)
        inten = inten.copy()
        inten[mask0] = 0.0

    if remove_low_intensity and remove_low_intensity > 0:
        keep = inten >= float(remove_low_intensity)
        pos3 = pos3[keep]
        inten = inten[keep]

    return _rasterize_spots_to_image(
        positions_xy_ainv=pos3[:, :2],
        intensities=inten,
        shape=shape,
        beam_pos_rc=beam_pos_rc,
        g_per_pixel=g_per_pixel,
        sigma_pix=sigma_pix,
        in_plane_angle_deg=in_plane_angle_deg,
    )


WAVELENGTH_A = electron_wavelength(VOLTAGE_KV)
CLEN_M = clen_for_dmin(MAX_RESOLUTION)
PIXELS_PER_M = 17_857.14285714286
IN_PLANE_ANGLE_DEG = 180
SIGMA_PIX = 1
FAST_MODE = False
NORMALISE = True
FAST_CLIP_TH = 1e-12
INTENSITY_SCALE = 10_000

working_dir = REF_PATH.parent
create_empty_backgrounds(working_dir, sims.current_size)
empty_h5 = working_dir / f"{sims.current_size}_empty_backgrounds.h5"

sim_folder = get_next_simulation_folder(working_dir)
sim_folder.mkdir(exist_ok=True)

cell_path = sim_folder / f"{REF_PATH.stem}.cell"
geom_path = sim_folder / f"{REF_PATH.stem}.geom"
h5_path = sim_folder / "sim.h5"
sol_path = sim_folder / "orientation_matrices.sol"

centering = phase.space_group.short_name[0].upper()
if centering in {"A", "C"}:
    centering = "S"

write_cell_file(
    {
        "lattice_type": phase.space_group.crystal_system.lower(),
        "centering": centering,
        "a": phase.structure.lattice.a,
        "b": phase.structure.lattice.b,
        "c": phase.structure.lattice.c,
        "alpha": phase.structure.lattice.alpha,
        "beta": phase.structure.lattice.beta,
        "gamma": phase.structure.lattice.gamma,
    },
    cell_path,
)
write_geom_file(geom_path, wavelength=WAVELENGTH_A, clen=CLEN_M, res=PIXELS_PER_M)

copy_h5_file(empty_h5, h5_path)

calibration = calculate_calibration(
    wavelength_A=WAVELENGTH_A,
    clen_m=CLEN_M,
    res=PIXELS_PER_M,
)
B_mat = compute_B(
    (
        phase.structure.lattice.a,
        phase.structure.lattice.b,
        phase.structure.lattice.c,
        phase.structure.lattice.alpha,
        phase.structure.lattice.beta,
        phase.structure.lattice.gamma,
    )
)

with h5py.File(h5_path, "r+", libver="latest") as f:
    imgs = f["entry/data/images"]
    ori = f["entry/data"].require_dataset(
        "simulation_orientation_matrices",
        shape=(imgs.shape[0], 3, 3),
        dtype=float,
    )
    thickness_nm_ds = f["entry/data"].require_dataset(
        "simulation_thickness_nm",
        shape=(imgs.shape[0],),
        dtype=float,
    )
    thickness_A_ds = f["entry/data"].require_dataset(
        "simulation_thickness_A",
        shape=(imgs.shape[0],),
        dtype=float,
    )
    det_shift_x_mm = f["entry"]["data"].require_dataset(
        "det_shift_x_mm",
        shape=(imgs.shape[0],),
        dtype=float,
        fillvalue=0,
    )
    det_shift_y_mm = f["entry"]["data"].require_dataset(
        "det_shift_y_mm",
        shape=(imgs.shape[0],),
        dtype=float,
        fillvalue=0,
    )

    shape = imgs.shape[-2:]
    beam_pos = (shape[0] // 2, shape[1] // 2)
    gpp = _g_per_pixel(WAVELENGTH_A, CLEN_M, PIXELS_PER_M)

    meta = f["entry"].attrs
    meta.update(
        ref_file=str(REF_PATH),
        resolution_deg=RESOLUTION_DEG,
        max_resolution=MAX_RESOLUTION,
        direct_beam=DIRECT_BEAM,
        beam_position=beam_pos,
        max_excitation_err=MAX_EXC_ERR_AINV,
        shape_factor_width=SHAPE_FACTOR_W_AINV,
        voltage_kV=VOLTAGE_KV,
        wavelength_A=WAVELENGTH_A,
        clen_m=CLEN_M,
        pixels_per_m=PIXELS_PER_M,
        calibration=calibration,
        in_plane_angle_deg=IN_PLANE_ANGLE_DEG,
        sigma_pix=SIGMA_PIX,
        fast_mode=FAST_MODE,
        normalise=NORMALISE,
        fast_clip_th=FAST_CLIP_TH,
        intensity_scale=INTENSITY_SCALE,
        use_abtem_multislice=USE_ABTEM_MULTISLICE,
        thickness_source=str(THICKNESS_SOURCE),
        thickness_mean_nm=float(thickness_nm.mean()),
        thickness_std_nm=float(thickness_nm.std()),
        thickness_min_nm=float(thickness_nm.min()),
        thickness_max_nm=float(thickness_nm.max()),
        abtem_sampling_A=float(ABTEM_SAMPLING_A),
        abtem_slice_thickness_A=float(ABTEM_SLICE_THICKNESS_A),
        abtem_projection=str(ABTEM_PROJECTION),
        abtem_parametrization=str(ABTEM_PARAMETRIZATION),
        abtem_remove_low_intensity=float(ABTEM_REMOVE_LOW_INTENSITY),
    )

    thickness_nm_ds[:] = thickness_nm
    thickness_A_ds[:] = thickness_nm * 10.0

    for i in tqdm(range(imgs.shape[0]), desc="Writing patterns"):
        if USE_ABTEM_MULTISLICE:
            frame_thickness_A = float(thickness_nm[i] * 10.0)
            pattern = _abtem_multislice_pattern(
                ref_path=REF_PATH,
                rotation_matrix_3x3=sims.rotations[i].to_matrix().squeeze(),
                thickness_A=frame_thickness_A,
                voltage_kV=float(VOLTAGE_KV),
                max_resolution_A=float(MAX_RESOLUTION),
                direct_beam=bool(DIRECT_BEAM),
                sampling_A=float(ABTEM_SAMPLING_A),
                slice_thickness_A=float(ABTEM_SLICE_THICKNESS_A),
                projection=str(ABTEM_PROJECTION),
                parametrization=str(ABTEM_PARAMETRIZATION),
                remove_low_intensity=float(ABTEM_REMOVE_LOW_INTENSITY),
                shape=shape,
                beam_pos_rc=beam_pos,
                g_per_pixel=gpp,
                sigma_pix=float(SIGMA_PIX),
                in_plane_angle_deg=float(IN_PLANE_ANGLE_DEG),
            )
        else:
            pattern = sims.irot[i].get_diffraction_pattern(
                shape=shape,
                direct_beam_position=beam_pos,
                in_plane_angle=IN_PLANE_ANGLE_DEG,
                sigma=SIGMA_PIX,
                calibration=calibration,
                fast=FAST_MODE,
                normalize=NORMALISE,
                fast_clip_threshold=FAST_CLIP_TH,
            )

        imgs[i] += (pattern * INTENSITY_SCALE).astype(imgs.dtype)
        ori[i] = B_mat @ sims.rotations[i].to_matrix().squeeze()

LAT_CODE = {
    "triclinic": "a",
    "monoclinic": "m",
    "orthorhombic": "o",
    "tetragonal": "t",
    "rhombohedral": "h",
    "hexagonal": "h",
    "cubic": "c",
}
AXIS_MAP = {
    "monoclinic": "b",
    "tetragonal": "c",
    "hexagonal": "c",
    "trigonal": "c",
    "rhombohedral": "c",
}

unique = AXIS_MAP.get(phase.space_group.crystal_system.lower())
if unique:
    bravais = f"{LAT_CODE[phase.space_group.crystal_system.lower()]}{centering}{unique}"
else:
    bravais = f"{LAT_CODE[phase.space_group.crystal_system.lower()]}{centering}"

with h5py.File(h5_path, "r") as fh:
    ori = fh["entry/data/simulation_orientation_matrices"][:]

with open(sol_path, "w") as fh:
    for idx, m in enumerate(ori):
        line = " ".join(f"{v:+.7f}" for v in m.flatten())
        fh.write(f"{h5_path} //{idx} {line} 0.000 0.000 {bravais}\n")

logging.info("All files written to %s", sim_folder)

# 3. Run Gandalf integration from the generated .sol

In [ ]:
# %%
from gandalf_iterator import gandalf_iterator

extra_flags = [
    "--no-revalidate", "--no-half-pixel-shift",
    "--peaks=peakfinder9",
    "--indexing=file", f"--fromfile-input-file={sol_path}",
    "--no-check-cell", "--no-check-peaks", "--no-retry", "--no-refine",
    "--integration=rings", "--int-radius=4,5,7",
    "--no-non-hits-in-stream", "--fix-profile-radius=70000000",
]

cell_path = str(cell_path)
geom_path = str(geom_path)
gandalf_iterator(
    geomfile_path=geom_path,
    cellfile_path=cell_path,
    input_path=sim_folder,
    output_file_base="from_file",
    num_threads=24,
    x=512.5,
    y=512.5,
    step=0.5,
    layers=0,
    extra_flags=extra_flags,
)
logging.info("Gandalf integration finished.")